# 05 – Train Models

Trains all comparison models (Naive, v_train baseline, OLS, Ridge numeric, Ridge + categorical, LightGBM)
on the `20260314` feature set. Models and results are saved to `data/04_model_outputs/20260314/`.

**Prerequisites:** `03_feature_engineering.ipynb` must have produced `data/03_features/20260314/region_features.parquet`.

In [ ]:
import sys
from pathlib import Path

# Walk up to backend root (works regardless of notebook cwd)
_here = Path.cwd()
while _here.name != 'backend' and _here.parent != _here:
    _here = _here.parent
if str(_here) not in sys.path:
    sys.path.insert(0, str(_here))

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import src.paths as PATHS
from src.model.export_utils import save_model_bundle

SEED = 42
EXPERIMENT = '20260314'

## 1. Load features

In [ ]:
FEAT_PATH = PATHS.DATA_DIR / '03_features' / EXPERIMENT / 'region_features.parquet'
feat = pd.read_parquet(FEAT_PATH)

train = feat[feat['split'] == 'train'].copy()
test  = feat[feat['split'] == 'test'].copy()

TARGET = 'v_test'

print(f'Train: {len(train):,}  |  Test: {len(test):,}')
print(f'Target – train  mean={train[TARGET].mean():.3f}  std={train[TARGET].std():.3f}')
print(f'Target – test   mean={test[TARGET].mean():.3f}  std={test[TARGET].std():.3f}')
feat.head(3)

## 2. Feature sets

In [ ]:
FEATS_2 = ['v_train']

FEATS_3 = [
    'v_train', 'dist_t2',
    'train_span_yr', 'test_span_yr',
    'n_events_t1', 'max_rise_rate_t1', 'drawdown_index_t1', 'flood_days_t1',
    'n_events_t2', 'max_rise_rate_t2', 'drawdown_index_t2', 'flood_days_t2',
    'bend_exposure_n5', 'bend_exposure_n8',
]

FEATS_4 = FEATS_3 + [
    'is_nvo', 'river_enc', 'vegetation_class_enc', 'soil_group_enc', 'land_use_enc',
]

FEATS_LGB = [
    'v_train', 'dist_t2',
    'train_span_yr', 'test_span_yr',
    'erosion_vol_rate_t1',
    'n_events_t1', 'max_rise_rate_t1', 'drawdown_index_t1', 'flood_days_t1',
    'n_events_t2', 'max_rise_rate_t2', 'drawdown_index_t2', 'flood_days_t2',
    'bend_exposure_n5', 'bend_exposure_n8',
    'is_nvo', 'river_enc', 'vegetation_class_enc', 'soil_group_enc', 'land_use_enc',
]

CAT_FEATS = ['river_enc', 'vegetation_class_enc', 'soil_group_enc', 'land_use_enc']

print('Feature sets:')
print(f'  FEATS_2  ({len(FEATS_2):>2}): {FEATS_2}')
print(f'  FEATS_3  ({len(FEATS_3):>2}): {FEATS_3}')
print(f'  FEATS_4  ({len(FEATS_4):>2}): {FEATS_4}')
print(f'  FEATS_LGB({len(FEATS_LGB):>2}): {FEATS_LGB}')

## 3. Evaluation helpers

In [ ]:
RESULTS = {}   # keyed by model name
TAIL_THRESHOLD = 2.0  # m/yr — operationally critical regions

def root_mean_squared_error(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def evaluate(name, y_train_true, y_train_pred, y_test_true, y_test_pred):
    """Compute metrics and store in RESULTS. MAE primary, RMSE secondary, tail MAE for v_test > 2 m/yr."""
    y_true = np.asarray(y_test_true)
    y_pred = np.asarray(y_test_pred)
    mask = y_true > TAIL_THRESHOLD
    tail_n = int(mask.sum())
    tail_mae = mean_absolute_error(y_true[mask], y_pred[mask]) if tail_n > 0 else np.nan
    row = dict(
        train_rmse=root_mean_squared_error(y_train_true, y_train_pred),
        train_mae =mean_absolute_error(y_train_true, y_train_pred),
        train_r2  =r2_score(y_train_true, y_train_pred),
        test_rmse =root_mean_squared_error(y_test_true, y_test_pred),
        test_mae  =mean_absolute_error(y_test_true, y_test_pred),
        test_r2   =r2_score(y_test_true, y_test_pred),
        test_tail_mae=tail_mae,
        test_tail_n  =tail_n,
    )
    RESULTS[name] = row
    print(f"  {'metric':>14}    train       test")
    print(f"  {'MAE (prim)':>14}    {row['train_mae']:.4f}    {row['test_mae']:.4f}")
    print(f"  {'RMSE':>14}    {row['train_rmse']:.4f}    {row['test_rmse']:.4f}")
    print(f"  {'MAE tail>2':>14}    {'—':>6}    {row['test_tail_mae']:.4f}  (n={tail_n})")
    print(f"  {'R²':>14}    {row['train_r2']:.4f}    {row['test_r2']:.4f}")
    return row


def scatter_pred(y_true, y_pred, title, ax=None):
    if ax is None:
        _, ax = plt.subplots()
    ax.scatter(y_true, y_pred, s=4, alpha=0.3, linewidths=0)
    lim = max(abs(np.array(y_true).max()), abs(np.array(y_pred).max())) * 1.05
    ax.plot([-lim, lim], [-lim, lim], 'r--', lw=1)
    ax.set_xlabel('actual  (m/yr)')
    ax.set_ylabel('predicted  (m/yr)')
    ax.set_title(title, fontsize=9)


def prep_cats(df, cols):
    out = df[cols].copy()
    if 'is_nvo' in out.columns:
        out['is_nvo'] = out['is_nvo'].astype(int)
    return out.astype(float).values


def prep_lgb(df, cols):
    out = df[cols].copy()
    if 'is_nvo' in out.columns:
        out['is_nvo'] = out['is_nvo'].astype(int)
    return out.astype(float)

## 4. Model 0 – Naive mean

In [ ]:
global_mean = train[TARGET].mean()
y_train_pred_0 = np.full(len(train), global_mean)
y_test_pred_0  = np.full(len(test),  global_mean)

print('Model 0 – Naive mean')
evaluate('0 – Naive mean', train[TARGET], y_train_pred_0, test[TARGET], y_test_pred_0)

## 5. Model 1 – v_train baseline

In [ ]:
print('Model 1 – v_train baseline')
evaluate('1 – v_train baseline',
         train[TARGET], train['v_train'],
         test[TARGET],  test['v_train'])

## 6. Model 2 – OLS (v_train only)

In [ ]:
ols_vt = LinearRegression()
ols_vt.fit(train[FEATS_2].values, train[TARGET])

print('Model 2 – OLS v_train')
print(f'  coefficient: {ols_vt.coef_[0]:.4f}   intercept: {ols_vt.intercept_:.4f}')
evaluate('2 – OLS v_train',
         train[TARGET], ols_vt.predict(train[FEATS_2].values),
         test[TARGET],  ols_vt.predict(test[FEATS_2].values))

## 7. Model 3 – Ridge (numeric features)

In [ ]:
ridge_num = Pipeline([('scaler', StandardScaler()), ('ridge', Ridge(alpha=1.0))])
ridge_num.fit(train[FEATS_3].values, train[TARGET])

print('Model 3 – Ridge numeric')
for f, c in zip(FEATS_3, ridge_num.named_steps['ridge'].coef_):
    print(f'  {f:>26s}: {c:+.4f}')
evaluate('3 – Ridge numeric',
         train[TARGET], ridge_num.predict(train[FEATS_3].values),
         test[TARGET],  ridge_num.predict(test[FEATS_3].values))

## 8. Model 4 – Ridge (numeric + categorical)

In [ ]:
ridge_cat = Pipeline([('scaler', StandardScaler()), ('ridge', Ridge(alpha=1.0))])
ridge_cat.fit(prep_cats(train, FEATS_4), train[TARGET])

print('Model 4 – Ridge + categorical')
for f, c in zip(FEATS_4, ridge_cat.named_steps['ridge'].coef_):
    print(f'  {f:>30s}: {c:+.4f}')
evaluate('4 – Ridge + categorical',
         train[TARGET], ridge_cat.predict(prep_cats(train, FEATS_4)),
         test[TARGET],  ridge_cat.predict(prep_cats(test,  FEATS_4)))

## 9. Model 5 – LightGBM

In [ ]:
X_tr_lgb = prep_lgb(train, FEATS_LGB)
X_te_lgb = prep_lgb(test,  FEATS_LGB)

dtrain = lgb.Dataset(X_tr_lgb, label=train[TARGET],
                     categorical_feature=CAT_FEATS, free_raw_data=False)

lgb_params = dict(
    objective='regression',
    metric='rmse',
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    verbose=-1,
    seed=SEED,
)

lgb_model = lgb.train(
    lgb_params,
    dtrain,
    num_boost_round=500,
    callbacks=[lgb.log_evaluation(period=0)],
)

y_tr_lgb = lgb_model.predict(X_tr_lgb)
y_te_lgb = lgb_model.predict(X_te_lgb)

print('Model 5 – LightGBM')
evaluate('5 – LightGBM', train[TARGET], y_tr_lgb, test[TARGET], y_te_lgb)

## 10. Summary table

In [ ]:
summary = pd.DataFrame(RESULTS).T
summary = summary[['test_mae', 'test_rmse', 'test_tail_mae', 'test_r2', 'train_mae', 'train_rmse', 'train_r2']]

print('\n=== Model comparison – 20260314 feature set ===')
display(summary.round(4).style
        .background_gradient(subset=['test_mae'], cmap='RdYlGn_r')
        .background_gradient(subset=['test_tail_mae'], cmap='RdYlGn_r')
        .background_gradient(subset=['test_r2'], cmap='RdYlGn')
        .format('{:.4f}'))

fig, ax = plt.subplots(figsize=(9, 3.5))
colors = ['#2166ac' if 'LightGBM' in n else '#aec6e8' for n in summary.index]
ax.barh(summary.index, summary['test_mae'], color=colors, edgecolor='white')
ax.axvline(summary.loc['0 – Naive mean', 'test_mae'], color='grey',
           linestyle='--', lw=1, label='naive mean floor')
ax.set_xlabel('Test MAE  (m/yr) — primary metric')
ax.set_title(f'Test MAE — model comparison ({EXPERIMENT})', fontweight='bold')
ax.invert_yaxis()
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 11. Feature importance (LightGBM)

In [ ]:
fi = (pd.Series(lgb_model.feature_importance(importance_type='gain'), index=FEATS_LGB)
      .sort_values(ascending=True))

fig, ax = plt.subplots(figsize=(7, 5))
fi.plot.barh(ax=ax, color='#2166ac')
ax.set_xlabel('Feature importance (gain)')
ax.set_title(f'LightGBM feature importance ({EXPERIMENT})', fontweight='bold')
plt.tight_layout(); plt.show()

## 12. Ridge coefficient plot

In [ ]:
coef = pd.Series(ridge_cat.named_steps['ridge'].coef_, index=FEATS_4).sort_values()
colors_coef = ['#d73027' if c < 0 else '#4575b4' for c in coef]

fig, ax = plt.subplots(figsize=(7, 5))
coef.plot.barh(ax=ax, color=colors_coef)
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Standardised coefficient')
ax.set_title(f'Ridge (numeric + categorical) coefficients ({EXPERIMENT})', fontweight='bold')
plt.tight_layout(); plt.show()

## 13. Comparison with old models (20260312)

In [ ]:
from src.model.export_utils import load_model_bundle

old_bundle = load_model_bundle(PATHS.DATA_DIR / '04_model_outputs' / '20260312')
old_results = pd.DataFrame(old_bundle['results']).T

new_results = summary.copy()

compare_cols = ['test_mae', 'test_rmse', 'test_r2']
old_r = old_results[compare_cols].rename(columns=lambda c: f'{c}_old')
new_r = new_results[compare_cols].rename(columns=lambda c: f'{c}_new')

shared_models = old_r.index.intersection(new_r.index)
comparison = pd.concat([old_r.loc[shared_models], new_r.loc[shared_models]], axis=1)
comparison['test_mae_Δ'] = comparison['test_mae_new'] - comparison['test_mae_old']
comparison['test_r2_Δ']  = comparison['test_r2_new']  - comparison['test_r2_old']

print('Δ = new – old   (negative MAE delta = improvement)')
display(comparison[['test_mae_old', 'test_mae_new', 'test_mae_Δ',
                     'test_r2_old',  'test_r2_new',  'test_r2_Δ']].round(4)
        .style.background_gradient(subset=['test_mae_Δ'], cmap='RdYlGn_r')
               .background_gradient(subset=['test_r2_Δ'],  cmap='RdYlGn')
               .format('{:.4f}'))

## 14. Save models to `04_model_outputs/20260314/`

In [ ]:
OUTPUT_DIR = PATHS.DATA_DIR / '04_model_outputs' / EXPERIMENT

config = dict(
    TARGET=TARGET,
    FEATS_2=FEATS_2,
    FEATS_3=FEATS_3,
    FEATS_4=FEATS_4,
    FEATS_LGB=FEATS_LGB,
    CAT_FEATS=CAT_FEATS,
)

saved = save_model_bundle(
    path=OUTPUT_DIR,
    models={
        'ols':       ols_vt,
        'ridge_num': ridge_num,
        'ridge_cat': ridge_cat,
        'lgb':       lgb_model,
    },
    config=config,
    results=RESULTS,
)

print(f'Models saved to: {saved}')
for f in sorted(saved.iterdir()):
    print(f'  {f.name:40s}  {f.stat().st_size / 1e3:>8.1f} KB')